In [0]:
# Databricks notebook source
import requests
import datetime as dt
import uuid
from typing import Optional, Dict, List
from pyspark.sql import functions as F
import time 

# ---- Inputs ----
FINGRID_API_KEY = dbutils.secrets.get("fingrid", "api-key")

datasets: List[Dict] = [
    {"name": "fingrid_solar_power_generation_forecast_updated_every_15_minutes", "id": 248},
    {"name": "fingrid_wind_power_generation_forecast_updated_every_15_minutes",  "id": 245},
    {"name": "Electricity_consumption_by_customer_type",                         "id": 358},
]

BASE_URL = "https://data.fingrid.fi/api"
LANDING_ROOT = "s3a://wartsila-datalake-dev-landing/"

CATALOG = "w_dev"
SCHEMA  = "landing_admin"
WATERMARK_TABLE = f"{CATALOG}.{SCHEMA}.fingrid_watermarks"

INITIAL_START: Optional[str] = None   # e.g., "2015-01-01T00:00:00Z"
INITIAL_END:   Optional[str] = None   # None pulls up to now

# ---- Helpers ----
def ensure_watermark_table():
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {WATERMARK_TABLE} (
      dataset_id INT,
      dataset_name STRING,
      max_end_time TIMESTAMP
    )
    USING DELTA

    """)

def get_watermark_iso(dataset_id: int, dataset_name: str) -> str:
    row = (spark.table(WATERMARK_TABLE)
           .where((F.col("dataset_id")==dataset_id) & (F.col("dataset_name")==dataset_name))
           .select("max_end_time").first())
    if not row or row.max_end_time is None:
        base = INITIAL_START if INITIAL_START else "2025-09-01T00:00:00Z"
        return base
    return row.max_end_time.replace(tzinfo=dt.timezone.utc).isoformat().replace("+00:00","Z")

def update_watermark(dataset_id: int, dataset_name: str, new_max_end_iso: str):
    spark.sql(f"""
      MERGE INTO {WATERMARK_TABLE} t
      USING (
        SELECT {dataset_id} AS dataset_id,
               '{dataset_name}' AS dataset_name,
               timestamp('{new_max_end_iso.replace('Z','')}') AS max_end_time
      ) s
      ON t.dataset_id = s.dataset_id AND t.dataset_name = s.dataset_name
      WHEN MATCHED AND s.max_end_time > t.max_end_time THEN UPDATE SET max_end_time = s.max_end_time
      WHEN NOT MATCHED THEN INSERT (dataset_id, dataset_name, max_end_time)
           VALUES (s.dataset_id, s.dataset_name, s.max_end_time)
    """)

def build_url(dataset_id: int, page: Optional[int], start_iso: Optional[str], end_iso: Optional[str]) -> str:
    endpoint = f"/datasets/{dataset_id}/data"
    url = BASE_URL.rstrip("/") + endpoint
    params = []
    if start_iso: params.append(f"startTime={start_iso}")
    if end_iso:   params.append(f"endTime={end_iso}")
    if page:      params.append(f"page={page}")
    if params:    url += "?" + "&".join(params)
    return url

def write_payload(dataset_name: str, dataset_id: int, payload_text: str, page_num: int) -> str:
    ing_dt = dt.datetime.utcnow().strftime("%Y-%m-%d")
    fname = f"{ing_dt}-{page_num:06d}-{uuid.uuid4()}.json"
    subfolder = f"fingrid/{dataset_name}/dataset_{dataset_id}/"
    path_dir = f"{LANDING_ROOT}{subfolder}ingest_dt={ing_dt}/page={page_num:06d}/"
    dbutils.fs.mkdirs(path_dir)
    path = path_dir + fname
    dbutils.fs.put(path, payload_text, overwrite=False)
    return path

# ---- Run ----
ensure_watermark_table()
headers = {"x-api-key": FINGRID_API_KEY}

for ds in datasets:
    ds_name = ds["name"]
    ds_id   = ds["id"]
    print(f"Starting ingestion for dataset {ds_id} - {ds_name}")

    start_iso = get_watermark_iso(ds_id, ds_name)
    end_iso   = INITIAL_END

    page = 1
    total_written = 0
    max_end_seen: Optional[str] = None

    while True:
        # --- Add a delay before making the next request ---
        time.sleep(2) # Sleeps for 2 seconds to respect the 1 call/2s limit

        url = build_url(ds_id, page=page, start_iso=start_iso, end_iso=end_iso)
        
        try:
            resp = requests.get(url, headers=headers, timeout=30)
            resp.raise_for_status()
        except requests.exceptions.HTTPError as e:
            print(f"HTTP Error for URL {url}: {e}")
            # Optional: Implement more robust error handling like exponential backoff
            if e.response.status_code == 429:
                print("Rate limit exceeded. Waiting longer before retrying...")
                time.sleep(10) # Wait for 10 seconds before the next attempt
                continue # Retry the same page
            else:
                # For other HTTP errors (like the previous 500 error), you might want to stop
                print(f"Stopping ingestion for dataset {ds_id} due to unrecoverable error.")
                break 

        payload_text = resp.text
        obj = resp.json()
        
        # Write raw page to landing
        _ = write_payload(ds_name, ds_id, payload_text, page)
        total_written += 1

        # Track max endTime
        for rec in obj.get("data", []):
            et = rec.get("endTime")
            if et and (not max_end_seen or et > max_end_seen):
                max_end_seen = et

        # Pagination
        nxt = obj.get("pagination", {}).get("nextPage")
        if not nxt:
            break
        page = int(nxt)

    if max_end_seen:
        update_watermark(ds_id, ds_name, max_end_seen)

    print(f"Finished {ds_id} - wrote {total_written} pages; new watermark: {max_end_seen or start_iso}")